In this Notebook, we want to create empirical Derrida plots. These are plots that relate the current normalised Hamming distance between two initial configurations (horizontal axis) to the normalised Hamming distance of those configurations in the next time step (vertical axis). If the resulting data point in the scatter plot is above the first diagonal, the Hamming distance is increasing (the damage is spreading). Otherwise, the damage remains the same, or even shrinks. Especially interesting is the behaviour around the origin, which is a measure for the sensitivity to initial conditions. The slope of the linear fit through the first couple of points close to the origin is called the Derrida coefficient.

IMPORTANT: the Derrida coefficient is related to the slope, but is not EQUAL to the slope! See for example "On Creativity of Elementary Cellular Automata" by Adamatzky et al. Try to read and understand the original Derrida paper ("Evolution of overlaps between con gurations in random Boolean networks")

In this notebook we will
1. [CHECK] Define a function that creates two pairs of arrays that can be used to plot the Derrida plot
2. [CHECK] Define a function that can use these arrays to output a single Derrida coefficient
3. [CHECK] Define a function that can output the equivalent rule for isomorphically defined LLNAs, and use this function to create a subset of non-equivalent rules.
4. [CHECK] For all non-equivalent rules of a particular resolution, and for a particular network, create a plot that relates the Derrida coefficient with genotype parameters (Hamming weight and Boolean sensitivity). Note that we should only run over non-equivalent rules.
    - The hope is that the Boolean sensitivity will correlate with the Derrida coefficient. [It does, almost perfectly]
5. For a number of particularly interesting rules, consider the influence of a network parameter on the Derrida coefficient.
    - We expect to see a sort of phase change when a particular average neighbourhood size is reached.
    - We hope to see a phase change when going from regular networks to random networks.
6. Experiment with Derrida equilibria and Derrida maxima (cause they're harder to predict than the Derrida coefficient)

Remaining questions and further research topics are listed below:
- Can we make an analytical approximation of the Derrida plot and/or coordinate?
- What is the relationship with the Jacobian and other established methods?
- I should add the possiblity of choosing the order of the nodes that are being affected by the defect! In that way, I can compare what happens to the overall behaviour if you change states in nodes with high PageRank (for example) first, compared to low PageRank first. Right now, random nodes are being chosen every time.
- Perhaps I could make a Derrida plot counterpart for networks, where I look at the evolution of the size of the affected subgraph (?)
- Instead of the slope of the linear fit, I could look at the angle (0 to 90) and normalise it to 0 to 1. Not sure how useful that would be, though.
- I should compare the Derrida coefficient with (at least) the Shannon entropy, the word length distribution (calculated on whole diagram), and the Lempel-Ziv complexity.
- I should probably double-check for with ECAs (some seem a bit odd)

# 1. Function that creates the Derrida arrays

First load some modules and functions

In [ ]:
# standard preamble for the Notebooks I use

import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcParams

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

from tqdm import tqdm

import sys, os

from llna.automata import LLNA
from llna.analysis import boolean_sens, hamming_weight, average_metric_over_degrees
from llna.analysis import calculate_derrida_coefficient, get_derrida_arrays, derrida_spline_and_roots, derrida_map_analytical
from llna.rules import binary_indices, return_equivalent_rule

%load_ext autoreload
%autoreload 2

Make a very simple LLNA and visualise it in the usual fashion.

In [ ]:
resolution = 3
born_if = [1]
survive_if = [0,2]
model = LLNA(resolution, x=born_if, y=survive_if, iso=True)
# model = LLNA(resolution, iso=True)
model.diagram(degree=2, plot_dist=True)

Now calculate the arrays required to make the Derrida plot

In [ ]:
# make a single ring graph and its edges
num_nodes = 1001  # Number of nodes
ring_graph = ig.Graph.Ring(num_nodes)
# edges
ring_graph.to_directed()
edges = tc.tensor(ring_graph.get_edgelist()).T
ring_graph.to_undirected()

# create 25 data points for each value of rho_t
points_per_rho = 5
num_init_configs = 5
init_configs = np.random.randint(0,2,size=(num_init_configs,num_nodes))
return_until_dens = 1

inputs, outputs = get_derrida_arrays(
    ring_graph,
    model,
    points_per_rho=points_per_rho,
    num_init_configs=None,
    init_configs=init_configs, 
    return_until_dens=return_until_dens
)

Create a Derrida plot, and show it alongside the spacetime diagrams and the defect propagation (as a sanity check). Note that these results should coincide exactly with the 36 non-equivalent outer-totalistic ECAs.

In [ ]:
# find original TEP
H = np.array(model.forward(edges, tc.tensor(init_configs), T=num_nodes-1), dtype=int)

# find TEP from defected initial configuration
halfway_idx = init_configs.shape[-1]//2
init_configs_defect = init_configs.copy()
init_configs_defect[:,halfway_idx] = 1 - init_configs_defect[:,halfway_idx]

# find TEP from defect initial configuration
H_defect = np.array(model.forward(edges, tc.tensor(init_configs_defect), T=num_nodes-1), dtype=int)
# find difference pattern
H_difference = (H + H_defect) % 2

# plot
fig, axs = plt.subplots(1,4,figsize=(14,6))
alpha=0.01

example_idx = 0
axs[0].scatter(inputs, outputs, alpha=alpha, s=1)
axs[1].imshow(H[example_idx], cmap='Greys')
axs[2].imshow(H_defect[example_idx], cmap='Greys')
axs[3].imshow(H_difference[example_idx], cmap='Greys')

axs[0].set_title('Derrida plot')
axs[1].set_title('Original')
axs[2].set_title('Defect')
axs[3].set_title('Difference')

axs[0].set_xlim([-0.05, 1.05])
axs[0].set_ylim([-0.05, 1.05])
for ax in axs: ax.set_aspect(1)

Does this make sense? Better check this for correspondence with ECAs! It looks like it should be wrong.
- R3B2S4
- B : [1], S : [2]: 
111 110 101 100 011 010 001 000
 1   0   0   1   0   0   1   0 = 2 + 16 + 128 = 146 (seems legit so far)

In [ ]:
points_per_rho = 2
num_init_configs = 3

num_nodes = 101  # Number of nodes
num_edges_per_new_node_list = [2] #, 4, 6, 8, 10]  # Number of edges each new node adds
ring_graph = ig.Graph.Ring(num_nodes)
return_until_dens = 1.0

# init_config = np.zeros(num_nodes)
init_configs = np.random.randint(0, 2, size=(num_init_configs, num_nodes))

inputs_list = []
outputs_list = []
for num_edges_per_new_node in tqdm(num_edges_per_new_node_list, total=len(num_edges_per_new_node_list)):
    # Generate Scale-Free Network (Barabási–Albert Model)
    # ba_graph = ig.Graph.Barabasi(n=num_nodes, m=num_edges_per_new_node, directed=False)
    inputs, outputs = get_derrida_arrays(ring_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
    inputs_list.append(inputs)
    outputs_list.append(outputs)

# find TEP for comparison
ring_graph.to_directed()
edges = tc.tensor(ring_graph.get_edgelist()).T
ring_graph.to_undirected()
H = np.array(model.forward(edges, tc.tensor(init_configs), T=num_nodes-1), dtype=int)

edge_def_idx = 50
init_configs_def = init_configs.copy()
init_configs_def[:,edge_def_idx] = 1-init_configs_def[:,edge_def_idx]
H_def = np.array(model.forward(edges, tc.tensor(init_configs_def), T=num_nodes-1), dtype=int)

# figure
fig, axs = plt.subplots(1,2,figsize=(10,5))
fontsize=18
alpha=1
markersize=4
axs[0].plot([0, 1], [0, 1], 'k--')
for inputs, outputs, num_edges_per_new_node in zip(inputs_list, outputs_list, num_edges_per_new_node_list):
    axs[0].scatter(inputs, outputs, alpha=alpha, s=markersize, label=fr"{num_edges_per_new_node} edges per new node")

axs[0].set_title(fr"Derrida plot for {model.__str__(latex=True)}", fontsize=fontsize)
axs[0].set_xlabel(fr"Defect fraction at $t$", fontsize=fontsize)
axs[0].set_ylabel(fr"Defect fraction at $t+1$", fontsize=fontsize)

legend = axs[0].legend()
for handle in legend.legend_handles:
    handle.set_alpha(1)  # Set marker transparency in legend
    handle.set_sizes([20])  # Change marker size in legend

axs[0].set_aspect(1)
axs[0].set_xlim([-0.05, 1.05])
axs[0].set_ylim([-0.05, 1.05])

# show cropped version of TEP
# axs[1].imshow(H[0, :edge_def_idx*2, :edge_def_idx*2])
axs[1].imshow((H[0, :edge_def_idx*2, :edge_def_idx*2] + H_def[0, :edge_def_idx*2, :edge_def_idx*2]) % 2, cmap='Greys', alpha=0.5)


In [ ]:
resolution = 3
model = LLNA(resolution, iso=True)
B_set, S_set = model.rule

B_set_equiv, S_set_equiv = return_equivalent_rule(resolution, B_set, S_set)
model_equiv = LLNA(resolution, x=B_set_equiv, y=S_set_equiv, iso=True)

fig, axs = plt.subplots(2,1, figsize=(12,4))
model.diagram(ax=axs[0])
model_equiv.diagram(ax=axs[1])

fig.tight_layout()

In [ ]:
betas = range(2**resolution)
sigmas = range(2**resolution)

nonequiv_rules = []
equiv_rules = []
for beta in betas:
    born_if = binary_indices(beta)
    for sigma in sigmas:
        survive_if = binary_indices(sigma)
        if (beta, sigma) not in equiv_rules:
            nonequiv_rules.append((beta, sigma))
        beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
        equiv_rules.append((beta_equiv, sigma_equiv))
len(nonequiv_rules)

In [ ]:
### NOTE: making this image takes a while

RUN_AGAIN = False

if RUN_AGAIN:
    # make network
    num_nodes = 501
    ring_graph = ig.Graph.Ring(num_nodes)

    # make initial condition
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    # make figure
    fig, axs = plt.subplots(6,6,figsize=(9,10))
    fontsize=14
    alpha=0.01
    markersize=2

    points_per_rho = 5
    return_until_dens = 1

    inputs_list = []
    outputs_list = []
    derrida_coefficients = []
    # create derrida array
    for idx, (beta, sigma) in tqdm(enumerate(nonequiv_rules), total=len(nonequiv_rules)):
        # make model
        born_if = binary_indices(beta)
        survive_if = binary_indices(sigma)
        model = LLNA(resolution, born_if, survive_if, iso=True)
        # calculate derrida arrays and coefficients
        inputs, outputs = get_derrida_arrays(ring_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
        derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
        # save to lists
        inputs_list.append(inputs)
        outputs_list.append(outputs)
        derrida_coefficients.append(derrida_coefficient)
        # plot
        idx_h = idx % 6
        idx_v = idx // 6
        axs[idx_h,idx_v].scatter(inputs, outputs, alpha=alpha, s=markersize)
        axs[idx_h,idx_v].plot([0,1], [0, derrida_coefficient], color='r', lw=1, alpha=0.5)
        axs[idx_h,idx_v].set_title(fr"{model.__str__(latex=True)}", fontsize=fontsize)

    for ax_row in axs:
        for ax in ax_row:
            ax.plot([0,1], [0,1], color='k', ls='--', lw=1, alpha=0.5)
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            ax.set_xticklabels([0,1])
            ax.set_yticklabels([0,1])
            ax.set_aspect(1)

    fig.suptitle(f"Derrida plots for all nonequivalent resolution-3 LLNAs on a N={num_nodes} ring network", fontsize=fontsize+4)
    fig.supxlabel(fr"Fraction of defects at time step $t$", fontsize=fontsize+4)
    fig.supylabel(fr"Fraction of defects at time step $t+1$", fontsize=fontsize+4)

    fig.tight_layout(rect=(0, 0, 1, 0.98))

    # plt.savefig("derrida_plots-nonequiv_outertotalistic_CAs.pdf")

Check all Derrida coefficients for a ring network for resolution three

In [ ]:
derrida_coefficients = []
BSs = []
HWs = []
points_per_rho = 5
num_init_configs = 5
return_until_dens = 0.06
for idx, (beta, sigma) in tqdm(enumerate(nonequiv_rules), total=len(nonequiv_rules)):
    # make model and get Derrida coefficients
    born_if = binary_indices(beta)
    survive_if = binary_indices(sigma)
    model = LLNA(resolution, born_if, survive_if, iso=True)
    inputs, outputs = get_derrida_arrays(ring_graph, model, points_per_rho=points_per_rho, num_init_configs=num_init_configs, return_until_dens=return_until_dens)
    derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
    derrida_coefficients.append(derrida_coefficient)
    # find BS and HW for simple ring network
    degree=2
    BS = boolean_sens(resolution, born_if, survive_if, degree)
    HW = hamming_weight(resolution, born_if, survive_if, degree)
    BSs.append(BS)
    HWs.append(HW)
HWs = np.array([HWs])
HWs[HWs>0.5] = 1-HWs[HWs>0.5]

fig, axs = plt.subplots(1,2,figsize=(12,6), sharey=True)
fontsize=14

axs[0].scatter(BSs, derrida_coefficients)
axs[1].scatter(HWs, derrida_coefficients)

axs[0].set_xlabel("Boolean sensitivity", fontsize=fontsize)
axs[0].set_ylabel("Derrida coefficient", fontsize=fontsize)
axs[0].set_title("Derrida coefficient vs Boolean sensitivity", fontsize=fontsize)

axs[1].set_xlabel("Normalised Hamming Weight", fontsize=fontsize)
axs[1].set_ylabel("Derrida coefficient", fontsize=fontsize)
axs[1].set_title("Derrida coefficient vs Hamming weight", fontsize=fontsize)

fig.suptitle("All nonequivalent resolution-3 LLNAs on a ring network", fontsize=fontsize)

Interesting! For ECAs, there is an almost one-to-one correlation between the Boolean sensivity and the Derrida coefficient. **why does that make sense**?
- Because this is essentially the definition of the Boolean sensitivity: You're looking at a whole bunch of neighbourhoods with a single defect, which is then propagated to the next stage. This is mentioned briefly in Fretter 2009 as well.
- This means that we do not really care that much about the Derrida coefficient. Perhaps we care more about
    - The Derrida equilibrium: where $d^0 = d^1 = d^*$
    - The maximum of the Derrida curve, where $d^1 = d^1_\text{max}$, and the corresponding $d^0$.

Now check Derrida coefficients for all resolution 5 rules and plot against BS and HW.

In [ ]:
### NOTE: making this image takes a while

RUN_AGAIN = False

if RUN_AGAIN:
    # choose resolution
    resolution = 7
    betas = range(2**resolution)
    sigmas = range(2**resolution)

    # make network
    num_nodes = 501
    ring_graph = ig.Graph.Ring(num_nodes)

    # make initial condition
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    points_per_rho = 5
    return_until_dens = 0.06

    inputs_list = []
    outputs_list = []
    derrida_coefficients = []
    equiv_rules_list = []
    # create derrida array
    for beta in tqdm(betas, total=2**resolution):
        for sigma in sigmas:
            if (beta, sigma) not in equiv_rules_list:
                # make model
                born_if = binary_indices(beta)
                survive_if = binary_indices(sigma)
                model = LLNA(resolution, born_if, survive_if, iso=True)
                # calculate derrida arrays and coefficients
                inputs, outputs = get_derrida_arrays(ring_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
                derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
                # save to lists
                inputs_list.append(inputs)
                outputs_list.append(outputs)
                derrida_coefficients.append(derrida_coefficient)
                # find equivalent rule and save to list
                beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
                equiv_rules_list.append((beta_equiv, sigma_equiv))

In [ ]:
HWs = []
BSs = []
degree = 2

# NOTE you have to reset the equiv_rules_list, because some rules are equivalent to themselves
equiv_rules_list = []
for beta in tqdm(betas, total=2**resolution):
    for sigma in sigmas:
        if (beta, sigma) not in equiv_rules_list:
            # make model
            born_if = binary_indices(beta)
            survive_if = binary_indices(sigma)
            HW = hamming_weight(resolution, born_if, survive_if, degree)
            BS = boolean_sens(resolution, born_if, survive_if, degree)
            HWs.append(HW)
            BSs.append(BS)
            # find equivalent rule and save to list
            beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
            equiv_rules_list.append((beta_equiv, sigma_equiv))

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12,6), sharey=True)
fontsize=14

axs[0].scatter(BSs, derrida_coefficients)
axs[1].scatter(HWs, derrida_coefficients)

axs[0].set_xlabel("Boolean sensitivity", fontsize=fontsize)
axs[0].set_ylabel("Derrida coefficient", fontsize=fontsize)
axs[0].set_title("Derrida coefficient vs Boolean sensitivity", fontsize=fontsize)

axs[1].set_xlabel("Normalised Hamming Weight", fontsize=fontsize)
axs[1].set_ylabel("Derrida coefficient", fontsize=fontsize)
axs[1].set_title("Derrida coefficient vs Hamming weight", fontsize=fontsize)

for ax in axs:
    ax.set_xlim([-0.05, 1.05])

n_nonequiv_rules = 2**(2*resolution-1) + 2**(resolution-1)
fig.suptitle(f"All {n_nonequiv_rules} nonequivalent resolution-{resolution} LLNAs on a ring network", fontsize=fontsize)

Clearly this just remains perfectly linearly dependent, so it's not terribly interesting. The relation with the Hamming weight is just mediated by means of the relation with the BS. Let's see what happens if we move to different networks.

In [ ]:
# create a range of networks using the Watts-Strogatz model

from igraph import Graph

# Define the number of nodes and the number of nearest neighbors
num_nodes = 500
k = 4  # Each node is connected to k nearest neighbors in ring topology

# Define the rewiring probabilities
rewiring_probs = np.linspace(0, 1, 11)

# Create the networks
networks = []
for p in rewiring_probs:
    ws_graph = Graph.Watts_Strogatz(dim=1, size=num_nodes, nei=k//2, p=p)
    networks.append(ws_graph)

In [ ]:
# run some models on these networks

# choose resolution
resolution = 3
betas = range(2**resolution)
sigmas = range(2**resolution)

# Derrida coefficients parameters
points_per_rho = 5
return_until_dens = 0.06
n_init_configs = 5

inputs_list_per_network = []
outputs_list_per_network = []
derrida_coefficients_per_network = []
for network in tqdm(networks, total=len(networks)):
    # make initial condition
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    inputs_list = []
    outputs_list = []
    derrida_coefficients = []
    equiv_rules_list = []
    # create derrida array
    for beta in betas:
        for sigma in sigmas:
            if (beta, sigma) not in equiv_rules_list:
                # make model
                born_if = binary_indices(beta)
                survive_if = binary_indices(sigma)
                model = LLNA(resolution, born_if, survive_if, iso=True)
                # calculate derrida arrays and coefficients
                inputs, outputs = get_derrida_arrays(network, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
                derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
                # save to lists
                inputs_list.append(inputs)
                outputs_list.append(outputs)
                derrida_coefficients.append(derrida_coefficient)
                # find equivalent rule and save to list
                beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
                equiv_rules_list.append((beta_equiv, sigma_equiv))
    inputs_list_per_network.append(inputs_list)
    outputs_list_per_network.append(outputs_list)
    derrida_coefficients_per_network.append(derrida_coefficients)

    # _=plt.boxplot(derrida_coefficients_per_network, labels=np.round(rewiring_probs,1))

In [ ]:
from functools import partial
metric_function_bs = partial(boolean_sens, norm_degree=True, iso=True)
metric_function_hw = partial(hamming_weight, norm=True, iso=True)

BS_list_per_network = []
HW_list_per_network = []
for network in tqdm(networks, total=len(networks)):
    BS_list = []
    HW_list = []
    equiv_rules_list = []
    # create derrida array
    for beta in betas:
        for sigma in sigmas:
            if (beta, sigma) not in equiv_rules_list:
                # make model
                born_if = binary_indices(beta)
                survive_if = binary_indices(sigma)
                BS = average_metric_over_degrees(resolution, born_if, survive_if, metric_function_bs, network)
                HW = average_metric_over_degrees(resolution, born_if, survive_if, metric_function_hw, network)
                BS_list.append(BS)
                HW_list.append(HW)
                # find equivalent rule and save to list
                beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
                equiv_rules_list.append((beta_equiv, sigma_equiv))
    BS_list_per_network.append(BS_list)
    HW_list_per_network.append(HW_list)

In [ ]:
fig, axs = plt.subplots(11,2, figsize=(6,30), sharey=True, sharex=True)
for i, ax_row in enumerate(axs):
    network = networks[i]
    derrida_coefficients = derrida_coefficients_per_network[i]
    BSs = BS_list_per_network[i]
    HWs = HW_list_per_network[i]
    ax_row[0].scatter(BSs, derrida_coefficients)
    ax_row[1].scatter(HWs, derrida_coefficients)
    ax_row[0].set_title(f"Rewiring probability: {np.round(rewiring_probs[i],2)},")
    ax_row[0].set_xlabel("Boolean sensitivity")
    ax_row[0].set_ylabel("Derrida coefficient")
    ax_row[1].set_xlabel("Normalised Hamming Weight")

fig.tight_layout()

Clearly the Derrida coefficients stays identical to the Boolean sensitivity. That of course makes sense: if there is only one defect in the entire configuration, that defect is only part of a small number of neighbourhoods. The average effect of the defect in this neighbourhood is essentially a Monte Carlo approach to finding the Boolean sensitivity.

Perhaps other properties of the Derrida plot are more interesting to investigate? There are two possibilities:
- The critical density, where $d^0 = d^1$
- The Derrida maximum (and the corresponding $d^0$)

It is possible to calculate these analytically also, however.

### Cubic splines to approximate the critical point and the maximum

First plot some examples for exploration

In [ ]:
### NOTE: making this image takes a while

RUN_AGAIN = False
SAVEFIG = False

if RUN_AGAIN:
    # make model
    resolution=7

    # make graph
    num_nodes = 500
    k = 2  # Each node is connected to k nearest neighbors in ring topology
    rewiring_prob = .5
    ws_graph = ig.Graph.Watts_Strogatz(dim=1, size=num_nodes, nei=k, p=rewiring_prob)

    # make initial condition
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    # make figure
    num_axs_h = 6
    num_axs_v = 6
    fig, axs = plt.subplots(num_axs_v,num_axs_h,figsize=(9,10))
    num_axs = axs.shape[0]*axs.shape[1]
    fontsize=14
    alpha=0.01
    markersize=2

    points_per_rho = 5
    return_until_dens = 1

    inputs_list = []
    outputs_list = []
    equilibria_list = []
    inputs_max_list = []
    outputs_max_list = []
    derrida_coefficients = []
    # create derrida array
    for idx in tqdm(range(num_axs), total=num_axs):
        # choose random model
        model = LLNA(resolution, iso=True)
        # calculate derrida arrays and coefficients
        inputs, outputs = get_derrida_arrays(ws_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
        derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
        # save to lists
        inputs_list.append(inputs)
        outputs_list.append(outputs)
        derrida_coefficients.append(derrida_coefficient)
        # create spline and find roots
        spline, equilibria = derrida_spline_and_roots(inputs, outputs, num_bins=20)
        inputs_spline = np.linspace(0, 1, num_nodes)
        outputs_spline = spline(inputs_spline)
        # find and append maximum and equilibria
        inputs_max = inputs_spline[np.argmax(outputs_spline)]
        outputs_max = np.max(outputs_spline)
        inputs_max_list.append(inputs_max)
        outputs_max_list.append(outputs_max)
        equilibria_list.append(equilibria)
        # find indices
        idx_h = idx % num_axs_v
        idx_v = idx // num_axs_v
        # plot
        axs[idx_h,idx_v].scatter(inputs, outputs, alpha=alpha, s=markersize)
        axs[idx_h,idx_v].plot([0,1], [0, derrida_coefficient], color='r', lw=1, alpha=0.5)
        axs[idx_h,idx_v].set_title(fr"{model.__str__(latex=True)}", fontsize=fontsize)
        axs[idx_h,idx_v].plot(inputs_spline, outputs_spline, 'g-')
        axs[idx_h,idx_v].scatter(equilibria, equilibria, color='black', s=100, marker='+', zorder=10)
        axs[idx_h,idx_v].scatter(inputs_max, outputs_max, color='black', s=100, marker='x', zorder=10)

    for ax_row in axs:
        for ax in ax_row:
            ax.plot([0,1], [0,1], color='k', ls='--', lw=1, alpha=0.5)
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            ax.set_xticklabels([0,1])
            ax.set_yticklabels([0,1])
            ax.set_aspect(1)

    fig.suptitle(f"Derrida plots for a selection of resolution-{resolution} LLNAs\non a Watts-Strogatz network (N={num_nodes}, p={rewiring_prob})", fontsize=fontsize+6)
    fig.supxlabel(fr"Fraction of defects at time step $t=0$", fontsize=fontsize+4)
    fig.supylabel(fr"Fraction of defects at time step $t=1$", fontsize=fontsize+4)

    fig.tight_layout(rect=(0, 0, 1, 0.98))

    if SAVEFIG:
        plt.savefig("derrida_plots-selection_of_outertotalistic_CAs-resolution7.pdf")

Let's do the same for all non-equivalent resolution-3 LLNAs on a ring graph (ECAs)

In [ ]:
betas = range(2**resolution)
sigmas = range(2**resolution)

nonequiv_rules = []
equiv_rules = []
for beta in betas:
    born_if = binary_indices(beta)
    for sigma in sigmas:
        survive_if = binary_indices(sigma)
        if (beta, sigma) not in equiv_rules:
            nonequiv_rules.append((beta, sigma))
        beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
        equiv_rules.append((beta_equiv, sigma_equiv))
len(nonequiv_rules)

In [ ]:
### NOTE: making this image takes a while

RUN_AGAIN = True
SAVEFIG = False

if RUN_AGAIN:
    # make model
    resolution=3

    # make graph
    ### CHANGE THIS TO ALTER COMPUTATION TIME
    num_nodes = 501
    ring_graph = ig.Graph.Ring(num_nodes)

    # make initial condition
    ### CHANGE THIS TO ALTER COMPUTATION TIME
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    # make figure
    num_axs_h = 6
    num_axs_v = 6
    fig, axs = plt.subplots(num_axs_v,num_axs_h,figsize=(9,10))
    num_axs = axs.shape[0]*axs.shape[1]
    fontsize=14
    alpha=0.01
    markersize=2

    ### CHANGE THIS TO ALTER COMPUTATION TIME
    points_per_rho = 5
    return_until_dens = 1

    inputs_list = []
    outputs_list = []
    equilibria_list = []
    inputs_max_list = []
    outputs_max_list = []
    BS_list = []
    HW_list = []
    derrida_coefficients = []
    # create derrida coefficients et al
    for idx, (beta, sigma) in tqdm(enumerate(nonequiv_rules), total=len(nonequiv_rules)):
        # make model
        born_if = binary_indices(beta)
        survive_if = binary_indices(sigma)
        model = LLNA(resolution, born_if, survive_if, iso=True)
        # calculate derrida arrays and coefficients
        inputs, outputs = get_derrida_arrays(ring_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
        derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
        # save to lists
        inputs_list.append(inputs)
        outputs_list.append(outputs)
        derrida_coefficients.append(derrida_coefficient)
        # create spline and find roots
        spline, equilibria = derrida_spline_and_roots(inputs, outputs, num_bins=20)
        inputs_spline = np.linspace(0, 1, num_nodes)
        outputs_spline = spline(inputs_spline)
        # find and append maximum and equilibria
        inputs_max = inputs_spline[np.argmax(outputs_spline)]
        outputs_max = np.max(outputs_spline)
        inputs_max_list.append(inputs_max)
        outputs_max_list.append(outputs_max)
        equilibria_list.append(equilibria)
        # find the Boolean sensitivity and the Hamming weight
        BS = boolean_sens(resolution, born_if, survive_if, degree=2, norm_degree=True, iso=True)
        HW = hamming_weight(resolution, born_if, survive_if, degree=2)
        BS_list.append(BS)
        HW_list.append(HW)
        # find indices
        idx_h = idx % num_axs_v
        idx_v = idx // num_axs_v
        # plot
        alpha_factor = 100
        alpha = alpha_factor / num_nodes / points_per_rho / n_init_configs
        axs[idx_h,idx_v].scatter(inputs, outputs, alpha=alpha, s=markersize)
        axs[idx_h,idx_v].plot([0,1], [0, derrida_coefficient], color='r', lw=1)
        axs[idx_h,idx_v].set_title(fr"{model.__str__(latex=True)}", fontsize=fontsize)
        axs[idx_h,idx_v].plot(inputs_spline, outputs_spline, 'g-')
        axs[idx_h,idx_v].scatter(equilibria, equilibria, color='black', s=100, marker='+', zorder=10)
        axs[idx_h,idx_v].scatter(inputs_max, outputs_max, color='black', s=100, marker='x', zorder=10)

    for ax_row in axs:
        for ax in ax_row:
            ax.plot([0,1], [0,1], color='k', ls='--', lw=1, alpha=0.5)
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            ax.set_xticklabels([0,1])
            ax.set_yticklabels([0,1])
            ax.set_aspect(1)

    fig.suptitle(f"Derrida plots for a selection of resolution-{resolution} LLNAs\non a Ring network (N={num_nodes})", fontsize=fontsize+6)
    fig.supxlabel(fr"Fraction of defects at time step $t=0$", fontsize=fontsize+4)
    fig.supylabel(fr"Fraction of defects at time step $t=1$", fontsize=fontsize+4)

    fig.tight_layout(rect=(0, 0, 1, 0.98))

    if SAVEFIG:
        plt.savefig("derrida_plots-selection_of_outertotalistic_CAs-resolution3-with_equilibria_and_maxima.pdf")

In [ ]:
fig, axs = plt.subplots(1,5,figsize=(14,3))

axs[0].scatter(BS_list, derrida_coefficients)
axs[0].set_xlabel("Boolean sensitivity")
axs[0].set_ylabel("Derrida coefficient")
axs[0].set_title("Derrida coefficient\nvs Boolean sensitivity")
axs[0].set_xlim([-0.05, 1.05])

equilibria_first_elements = np.array([eq[0] if len(eq) > 0 else None for eq in equilibria_list])
axs[1].scatter(BS_list, equilibria_first_elements)
axs[1].set_xlabel("Boolean sensitivity")
axs[1].set_ylabel("Derrida equilibrium")
axs[1].set_title("Derrida equilibrium (if exists)\nvs Boolean sensitivity")
axs[1].set_xlim([-0.05, 1.05])

axs[2].scatter(BS_list, inputs_max_list)
axs[2].set_xlabel("Boolean sensitivity")
axs[2].set_ylabel(f"Derrida maximum ($d^0$)")
axs[2].set_title(f"Derrida maximum ($d^0$)\nvs Boolean sensitivity")
axs[2].set_xlim([-0.05, 1.05])
axs[2].set_ylim([-0.05, 1.05])

axs[3].scatter(BS_list, outputs_max_list)
axs[3].set_xlabel("Boolean sensitivity")
axs[3].set_ylabel(f"Derrida maximum ($d^1$)")
axs[3].set_title(f"Derrida maximum ($d^1$)\nvs Boolean sensitivity")
axs[3].set_xlim([-0.05, 1.05])
axs[3].set_ylim([-0.05, 1.05])

axs[4].scatter(inputs_max_list, outputs_max_list)
axs[4].set_xlabel("Derrida maximum ($d^0$)")
axs[4].set_ylabel(f"Derrida maximum ($d^1$)")
axs[4].set_title(f"Derrida maximum ($d^1$)\nvs Derrida maximum ($d^0$)")
axs[4].set_xlim([-0.05, 1.05])
axs[4].set_ylim([-0.05, 1.05])

fig.tight_layout()

There seems to be especially little correlation between the equilibrium and the Boolean sensitivity.

### Analytical calculation of Derrida curve

Observation: the Derrida map is essentially a generalised Boolean sensitivity. For calculating the Boolean sensitivity, we only consider the average change to the output when introducing a single defect. The Derrida map looks at what happens when we introduce all possible defects in the network. The 'generalised' Boolean sensitivity can generalise this to considering what happens on average to the update function when we introduce a fraction of $d^0$ defects.

Goal:
1. Find a well-argued formula for the Derrida curve
2. Compare these to the simulations (to be sure)
3. Take out equilibrium en maximum value
4. Compare these values with computational values (Lyapunov, entropy measures, etc)
5. Possible: look at how fast the real path diverges from the expected path (which is a measure for how fast clustering appears, I guess?)

- Let us say that we start off with an initial configuration with a density $\rho$.
- We make a copy of this configuration, and toggle some nodes such that the resulting normalised Hamming distance between the two configurations is $d^0$.
- Select a random node in the original configuration. This node has degree $k$. The probability of this node having $q$ activated (living) neighbours is $\dbinom{k}{q} \rho^q (1-\rho)^{k-q}$.
- Select the same node in the modified configuration. This node has $q'$ activated neighours:
$$
q' = q + t_{0\mapsto 1} - t_{1\mapsto 0} = q - t_{1\mapsto 0} + \delta,
$$
where $\delta$ is the total number of toggled nodes in this neighbourhood, and $t_{1\mapsto 0}$ is the number of node toggles that result in a dead cell (they're "toggled to death")
- The total number of toggled nodes is $\delta$, and is binomially distributed, with a probability $\dbinom{k}{\delta} (d^0)^\delta (1-d^0)^{k-\delta}$
- The number of node toggles that result in a dead cell is $t_{1\mapsto 0} = t$, and is hypergeometrically distributed, with a probability $\dfrac{\dbinom{q}{t} \dbinom{k-q}{\delta-t}}{\dbinom{k}{\delta}}$, where $t$ runs from $\max(0, \delta-(k-q))$ to $\min(q,\delta)$
- You can of course minimally toggle $0$ nodes and maximally toggle $\delta$ nodes.
- Interpretation of the lower bound: if the number of dead nodes $k-q$ is high (e.g. $\delta/2$), then you will run out of nodes to toggle to death, so it's impossible to have a low $t$ value.
- Interpretation of the upper bound: if the number of living cells $q$ is less than the number of nodes that are being toggled, it's unavoidable that some nodes will be toggled to death.

So, summing everything up, we have
$$
\underbrace{\sum_{q=0}^k \Big[\binom{k}{q} \rho^q (1-\rho)^{k-q}\Big]}_{\text{living nodes in original configuration}} \underbrace{\sum_{\delta=0}^k \Big[ \dbinom{k}{\delta} (d^0)^\delta (1-d^0)^{k-\delta} \Big]}_{\text{toggled nodes}} \underbrace{\sum_{t=\max(0, \delta-(k-q))}^{\min(q,\delta)} \Big[ {\binom{k}{\delta}}^{-1} \binom{q}{t} \binom{k-q}{\delta-t}\Big]}_{\text{toggled to death}} \underbrace{\Big[ \phi\left(s, \frac{q}{k}\right)\ \oplus \phi\left(s, \frac{q-t+\delta}{k}\right)\Big]}_{\text{count non-equal outcomes}}
$$

Now we add the probabilities for the central node.
$$
\sum_{q=0}^k P(q ~|~ k, \rho) \sum_{\delta=0}^k P(\delta ~|~ k, d^0) \sum_{t=t_\text{min}}^{t_\text{max}} P(t ~|~ k, \delta, q) \text{NEO}(k, \delta, q, t),
$$
where $\text{NEO}$ is an acronyn for "non-equal outcomes", defined as
$$
\begin{align*}
\text{NEO}(k, \delta, q, t) &= \rho d^0\Big[ \phi\left(1, \frac{q}{k}\right)\ \oplus \phi\left(0, \frac{q-t+\delta}{k}\right)\Big] + \rho (1-d^0)\Big[ \phi\left(1, \frac{q}{k}\right)\ \oplus \phi\left(1, \frac{q-t+\delta}{k}\right)\Big] + (1-\rho) d^0\Big[ \phi\left(0, \frac{q}{k}\right)\ \oplus \phi\left(1, \frac{q-t+\delta}{k}\right)\Big] + (1-\rho)(1-d^0)\Big[ \phi\left(0, \frac{q}{k}\right)\ \oplus \phi\left(0, \frac{q-t+\delta}{k}\right)\Big] \\
&= \sum_{s_1\in\{0,1\}} \sum_{s_2\in\{0,1\}} \rho^{s_1}(1-\rho)^{1-s_1}(d^0)^{1-s_2}(1-d^0)^{s_2} \Big[ \phi\left(s_1, \frac{q}{k}\right)\ \oplus \phi\left(s_2, \frac{q-t+\delta}{k}\right)\Big]
\end{align*}
$$

NOTE: we now follow the notes I made by hand (the formulas above still need to be changed)

In [ ]:
# verify whether it is correct.

### NOTE: making this image takes a while

RUN_AGAIN = True

resolution = 3
betas = range(2**resolution)
sigmas = range(2**resolution)
degree = 2

nonequiv_rules = []
equiv_rules = []
for beta in betas:
    born_if = binary_indices(beta)
    for sigma in sigmas:
        survive_if = binary_indices(sigma)
        if (beta, sigma) not in equiv_rules:
            nonequiv_rules.append((beta, sigma))
        beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
        equiv_rules.append((beta_equiv, sigma_equiv))
len(nonequiv_rules)

if RUN_AGAIN:
    # make network
    num_nodes = 501
    ring_graph = ig.Graph.Ring(num_nodes)

    # make initial condition
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    # make figure
    fig, axs = plt.subplots(6,6,figsize=(9,10))
    delta0_res = 101
    delta0_array = np.linspace(0,1,delta0_res)
    fontsize=14
    alpha=0.01
    markersize=2

    points_per_rho = 5
    return_until_dens = 1

    inputs_list = []
    outputs_list = []
    derrida_coefficients = []
    outputs_analytical_list = []
    # create derrida array
    for idx, (beta, sigma) in tqdm(enumerate(nonequiv_rules), total=len(nonequiv_rules)):
        # make model
        born_if = binary_indices(beta)
        survive_if = binary_indices(sigma)
        model = LLNA(resolution, born_if, survive_if, iso=True)
        # calculate derrida arrays and coefficients
        inputs, outputs = get_derrida_arrays(ring_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
        derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
        # calculate Derrida arrays analytically
        outputs_analytical = derrida_map_analytical(resolution, born_if, survive_if, delta0_array, degree, iso=True)
        # save to lists
        inputs_list.append(inputs)
        outputs_list.append(outputs)
        derrida_coefficients.append(derrida_coefficient)
        outputs_analytical_list.append(outputs_analytical)
        # plot
        idx_h = idx % 6
        idx_v = idx // 6
        axs[idx_h,idx_v].scatter(inputs, outputs, alpha=alpha, s=markersize)
        axs[idx_h,idx_v].plot([0,1], [0, derrida_coefficient], color='r', lw=1, alpha=0.5)
        axs[idx_h,idx_v].set_title(fr"{model.__str__(latex=True)}", fontsize=fontsize)
        axs[idx_h,idx_v].plot(delta0_array, outputs_analytical, color='maroon', lw=2, zorder=10)

    for ax_row in axs:
        for ax in ax_row:
            ax.plot([0,1], [0,1], color='k', ls='--', lw=1, alpha=0.5)
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            ax.set_xticklabels([0,1])
            ax.set_yticklabels([0,1])
            ax.set_aspect(1)

    fig.suptitle(f"Derrida plots for all nonequivalent resolution-3 LLNAs on a N={num_nodes} ring network", fontsize=fontsize+4)
    fig.supxlabel(fr"Fraction of defects at time step $t$", fontsize=fontsize+4)
    fig.supylabel(fr"Fraction of defects at time step $t+1$", fontsize=fontsize+4)

    fig.tight_layout(rect=(0, 0, 1, 0.98))

    # plt.savefig("derrida_plots-nonequiv_outertotalistic_CAs.pdf")

This looks right!

Let's do a sanity check. There are two cases for which calculating the defect should be easy:
1. When the defect is minimal: this means there is no difference, so the Derrida curve should _always_ go through the origin (which it does)
2. When the defect is maximal: this means that the configurations are each others complement. Let's see what happens then.

If every node is toggled, it means that every neighbourhood as a whole is toggled, such that 
1. The central node is toggled
2. The density of the neighbourhood is 'mirrored': $\rho_i \mapsto 1-\rho_i$

Toggling the central node is equivalent to taking the output after switching $B$ and $S$ sets: $B \leftrightarrow S$. Mirroring the neighbourhood density is equivalent to mapping the sets: $(B, S) \mapsto (\bar{B}, \bar{S})$. We want to look at the Hamming distance between the outputs of the truth table of both rules. This can be expressed by means of the mean-field parameters:

$$
\begin{align*}
M_{B,q} &= \sum_{\text{nbh states that sum to } q} \phi(0, q/k) = \binom{k}{q}\left[\frac{q}{k} \in \bigcup B\right] \\
M_{S,q} &= \sum_{\text{nbh states that sum to } q} \phi(1, q/k) = \binom{k}{q}\left[\frac{q}{k} \in \bigcup S\right],
\end{align*}
$$
Where the square brackets evaluate the truth of the statement and map to $0$ for false, and $1$ for true (an Iverson bracket).

>PS I could also write $\mathbf{1}_{\bigcup B}(q/k)$, using the so-called indicator function.

By the way, recall here that of course the normalised Hamming weight is simply
$$
\text{HW}_i = \frac{1}{2^{k_i + 1}}\sum_{q=0}^{k_i} \Big( M_{B,q} + M_{S,q}\Big).
$$

So, if we expect a uniform distribution over the neighbourhoods, the expected defect in the next timestep when the configurations are each other's complement is
$$
d^1(d^0 = 1) = \frac{1}{2^{k + 1}}\sum_{q=0}^{k} \binom{k}{q} \left(\left[\frac{q}{k} \in \bigcup B\right] \oplus \left[\frac{q}{k} \in \bigcup \bar{S}\right] + \left[\frac{q}{k} \in \bigcup S\right] \oplus \left[\frac{q}{k} \in \bigcup \bar{B}\right]\right)
$$
This can very easily be calculated directly from the local update rule. Let's give it a try.

In [ ]:
from math import comb
from llna.analysis import _interval_encoding
from scipy.stats import binom

def mean_field_param(resolution, born_or_survive_set, degree, nbh_sum, iso=True):
    """
    Calculate the mean field parameter for a given set of born or survive intervals.
    """
    # check whether input makes sense
    if born_or_survive_set and max(born_or_survive_set) >= resolution:
        raise ValueError(f"Resolution {resolution} is too small for the provided born or survive set.")
    if (nbh_sum < 0) or (nbh_sum > degree):
        raise ValueError(f"Neighbourhood sum must be between 0 and the degree ({degree}), not {nbh_sum}.")
    # calculate which interval the density sum falls in
    rho = nbh_sum/degree
    rho_interval = _interval_encoding(resolution, np.array([[rho]]), iso=iso)[0, 0].argmax()
    if rho_interval in born_or_survive_set:
        mfp = comb(degree, nbh_sum)
        return mfp
    return 0

def d1_from_complement(resolution, born_if, survive_if, degree, iso=True):
    # find mirror sets
    born_if_mirror = [resolution-1-i for i in born_if]
    survive_if_mirror = [resolution-1-i for i in survive_if]
    # get all possible nbh state densities
    rhos = np.linspace(0, 1, degree+1)
    # get all possible binomial factors
    binom_q = binom.pmf(np.arange(degree+1), degree, 0.5)
    # calculate which intervals the densities fall in
    rho_intervals = _interval_encoding(resolution, rhos[np.newaxis,:], iso=iso)[0].argmax(axis=1)
    # find out whether the densities are in the relevant set
    rho_in_born_if = np.array([rho in born_if for rho in rho_intervals])
    rho_in_survive_if_mirror = np.array([rho in survive_if_mirror for rho in rho_intervals])
    rho_in_survive_if = np.array([rho in survive_if for rho in rho_intervals])
    rho_in_born_if_mirror = np.array([rho in born_if_mirror for rho in rho_intervals])
    # Take XOR of the sets and turn into ints
    first_xor = np.array(rho_in_born_if ^ rho_in_survive_if_mirror, dtype=int)
    second_xor = np.array(rho_in_survive_if ^ rho_in_born_if_mirror, dtype=int)
    # sum with appropriate binomial factors. Divide by two to normalise appropriately
    d1 = np.sum(binom_q * (first_xor + second_xor)) / 2
    # clamp in the unit interval
    d1 = max(0., min(1., d1))
    return d1


In [ ]:
# Sanity check: does this add up to the normalised Hamming weight? YES IT DOES

resolution = 5
degree = 7
model = LLNA(resolution, iso=True)
B_set, S_set = model.rule

mfp_B = np.sum([mean_field_param(resolution, B_set, degree, nbh_sum, iso=True) for nbh_sum in range(degree+1)])
mfp_S = np.sum([mean_field_param(resolution, S_set, degree, nbh_sum, iso=True) for nbh_sum in range(degree+1)])
HW = hamming_weight(resolution, B_set, S_set, degree, norm=False, iso=True)

(mfp_B + mfp_S) == HW

OK, this seems to work out. Let's check whether this coincides with the actual Derrida plots.

In [ ]:
# verify whether it is correct.

### NOTE: making this image takes a while

RUN_AGAIN = True

resolution = 3
betas = range(2**resolution)
sigmas = range(2**resolution)
degree = 2

nonequiv_rules = []
equiv_rules = []
for beta in betas:
    born_if = binary_indices(beta)
    for sigma in sigmas:
        survive_if = binary_indices(sigma)
        if (beta, sigma) not in equiv_rules:
            nonequiv_rules.append((beta, sigma))
        beta_equiv, sigma_equiv = return_equivalent_rule(resolution, born_if, survive_if, return_decimals=True)
        equiv_rules.append((beta_equiv, sigma_equiv))
len(nonequiv_rules)

if RUN_AGAIN:
    # make network
    num_nodes = 501
    ring_graph = ig.Graph.Ring(num_nodes)

    # make initial condition
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    # make figure
    fig, axs = plt.subplots(6,6,figsize=(9,10))
    fontsize=14
    alpha=0.01
    markersize=2

    points_per_rho = 5
    return_until_dens = 1

    inputs_list = []
    outputs_list = []
    derrida_coefficients = []
    # inputs_analytical_list = []
    # outputs_analytical_list = []
    d1_from_comp_list = []
    # create derrida array
    for idx, (beta, sigma) in tqdm(enumerate(nonequiv_rules), total=len(nonequiv_rules)):
        # make model
        born_if = binary_indices(beta)
        survive_if = binary_indices(sigma)
        model = LLNA(resolution, born_if, survive_if, iso=True)
        # calculate derrida arrays and coefficients
        inputs, outputs = get_derrida_arrays(ring_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
        derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
        # calculate Derrida arrays analytically
        # inputs_analytical = np.linspace(0, 1, 101)
        # outputs_analytical = np.array([derrida_map_analytical(resolution, born_if, survive_if, degree, d0, iso=True) for d0 in np.linspace(0,1,101)])
        # find d1 from complementary configurations
        d1_from_comp = d1_from_complement(resolution, born_if, survive_if, degree, iso=True)
        # save to lists
        inputs_list.append(inputs)
        outputs_list.append(outputs)
        derrida_coefficients.append(derrida_coefficient)
        # inputs_analytical_list.append(inputs_analytical)
        # outputs_analytical_list.append(outputs_analytical)
        d1_from_comp_list.append(d1_from_comp)
        # plot
        idx_h = idx % 6
        idx_v = idx // 6
        axs[idx_h,idx_v].scatter(inputs, outputs, alpha=alpha, s=markersize)
        axs[idx_h,idx_v].plot([0,1], [0, derrida_coefficient], color='r', lw=1, alpha=0.5)
        axs[idx_h,idx_v].set_title(fr"{model.__str__(latex=True)}", fontsize=fontsize)
        # axs[idx_h,idx_v].plot(inputs_analytical, outputs_analytical, color='maroon', lw=2, zorder=10)
        axs[idx_h,idx_v].scatter(1, d1_from_comp, color='black', s=100, marker='x', zorder=10)

    for ax_row in axs:
        for ax in ax_row:
            ax.plot([0,1], [0,1], color='k', ls='--', lw=1, alpha=0.5)
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            ax.set_xticklabels([0,1])
            ax.set_yticklabels([0,1])
            ax.set_aspect(1)

    fig.suptitle(f"Derrida plots for all nonequivalent resolution-3 LLNAs on a N={num_nodes} ring network", fontsize=fontsize+4)
    fig.supxlabel(fr"Fraction of defects at time step $t$", fontsize=fontsize+4)
    fig.supylabel(fr"Fraction of defects at time step $t+1$", fontsize=fontsize+4)

    fig.tight_layout(rect=(0, 0, 1, 0.98))

    # plt.savefig("derrida_plots-nonequiv_outertotalistic_CAs.pdf")

Now let's calculate for higher resolutions

In [ ]:
### NOTE: making this image takes a while

RUN_AGAIN = True
SAVEFIG = False

if RUN_AGAIN:
    # make model
    resolution=7

    # make graph
    num_nodes = 500
    k = 2  # Each node is connected to k nearest neighbors in ring topology
    rewiring_prob = .5
    degree = 6
    regular_graph = ig.Graph.K_Regular(num_nodes, degree)
    delta0_array = np.linspace(0,1,51)

    # make initial condition
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    # make figure
    num_axs_h = 6
    num_axs_v = 6
    fig, axs = plt.subplots(num_axs_v,num_axs_h,figsize=(9,10))
    num_axs = axs.shape[0]*axs.shape[1]
    fontsize=14
    alpha=0.01
    markersize=2

    points_per_rho = 5
    return_until_dens = 1

    inputs_list = []
    outputs_list = []
    derrida_coefficients = []
    # create derrida array
    for idx in tqdm(range(num_axs), total=num_axs):
        # choose random model
        model = LLNA(resolution, iso=True)
        B_set, S_set = model.rule
        # calculate derrida arrays and coefficients
        inputs, outputs = get_derrida_arrays(regular_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
        derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
        # save to lists
        inputs_list.append(inputs)
        outputs_list.append(outputs)
        derrida_coefficients.append(derrida_coefficient)
        # create analytical Derrida plot
        delta1_array = derrida_map_analytical(resolution, B_set, S_set, delta0_array, degree)
        # find indices
        idx_h = idx % num_axs_v
        idx_v = idx // num_axs_v
        # plot
        axs[idx_h,idx_v].scatter(inputs, outputs, alpha=alpha, s=markersize)
        axs[idx_h,idx_v].plot([0,1], [0, derrida_coefficient], color='r', lw=1, alpha=0.5)
        axs[idx_h,idx_v].set_title(fr"{model.__str__(latex=True)}", fontsize=fontsize)
        axs[idx_h,idx_v].plot(delta0_array, delta1_array, 'k-')

    for ax_row in axs:
        for ax in ax_row:
            ax.plot([0,1], [0,1], color='k', ls='--', lw=1, alpha=0.5)
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            ax.set_xticklabels([0,1])
            ax.set_yticklabels([0,1])
            ax.set_aspect(1)

    fig.suptitle(f"Derrida plots for a selection of resolution-{resolution} LLNAs\non a Regular Network (N={num_nodes}, $ k = {degree}$)", fontsize=fontsize+6)
    fig.supxlabel(fr"Fraction of defects at time step $t=0$", fontsize=fontsize+4)
    fig.supylabel(fr"Fraction of defects at time step $t=1$", fontsize=fontsize+4)

    fig.tight_layout(rect=(0, 0, 1, 0.98))

    if SAVEFIG:
        plt.savefig("derrida_plots-selection_of_outertotalistic_CAs-resolution7.pdf")

## Let's do the same for a random graph

In [ ]:
### NOTE: making this image takes a while

RUN_AGAIN = True
SAVEFIG = False

if RUN_AGAIN:
    # make model
    resolution=7

    # make connected random graph
    num_nodes = 100
    degree = 6
    while True:
        random_graph = ig.Graph.Erdos_Renyi(n=num_nodes, m=num_nodes*degree//2)
        if random_graph.is_connected():
            break
    degree_per_node = random_graph.degree()
    degrees = np.arange(1,max(degree_per_node)+1)
    degree_counts = np.bincount(random_graph.degree())[1:]

    # initial defect array
    delta0_array = np.linspace(0,1,21)

    # make initial condition
    n_init_configs = 5
    init_configs = np.random.randint(0, 2, size=(n_init_configs, num_nodes))

    # make figure
    num_axs_h = 6
    num_axs_v = 6
    fig, axs = plt.subplots(num_axs_v,num_axs_h,figsize=(9,10))
    num_axs = axs.shape[0]*axs.shape[1]
    fontsize=14
    alpha=0.05
    markersize=2

    points_per_rho = 5
    return_until_dens = 1

    inputs_list = []
    outputs_list = []
    derrida_coefficients = []
    # create derrida array
    for idx in tqdm(range(num_axs), total=num_axs):
        # choose random model
        model = LLNA(resolution, iso=True)
        B_set, S_set = model.rule
        # calculate derrida arrays and coefficients
        inputs, outputs = get_derrida_arrays(random_graph, model, points_per_rho=points_per_rho, init_configs=init_configs, return_until_dens=return_until_dens)
        derrida_coefficient = calculate_derrida_coefficient(inputs, outputs)
        # save to lists
        inputs_list.append(inputs)
        outputs_list.append(outputs)
        derrida_coefficients.append(derrida_coefficient)
        # create analytical Derrida plot
        delta1_array = np.sum([degree_count*derrida_map_analytical(resolution, B_set, S_set, delta0_array, degree) for degree, degree_count in zip(degrees, degree_counts)], axis=0) / num_nodes
        # find indices
        idx_h = idx % num_axs_v
        idx_v = idx // num_axs_v
        # plot
        axs[idx_h,idx_v].scatter(inputs, outputs, alpha=alpha, s=markersize)
        axs[idx_h,idx_v].plot([0,1], [0, derrida_coefficient], color='r', lw=1, alpha=0.5)
        axs[idx_h,idx_v].set_title(fr"{model.__str__(latex=True)}", fontsize=fontsize)
        axs[idx_h,idx_v].plot(delta0_array, delta1_array, 'k-')

    for ax_row in axs:
        for ax in ax_row:
            ax.plot([0,1], [0,1], color='k', ls='--', lw=1, alpha=0.5)
            ax.set_xlim([-0.05, 1.05])
            ax.set_ylim([-0.05, 1.05])
            ax.set_xticks([0,1])
            ax.set_yticks([0,1])
            ax.set_xticklabels([0,1])
            ax.set_yticklabels([0,1])
            ax.set_aspect(1)

    fig.suptitle(f"Derrida plots for a selection of resolution-{resolution} LLNAs\non a Random Network (N={num_nodes}, $\\langle k \\rangle  = {degree}$)", fontsize=fontsize+6)
    fig.supxlabel(fr"Fraction of defects at time step $t=0$", fontsize=fontsize+4)
    fig.supylabel(fr"Fraction of defects at time step $t=1$", fontsize=fontsize+4)

    fig.tight_layout(rect=(0, 0, 1, 0.98))

    if SAVEFIG:
        plt.savefig("derrida_plots-selection_of_outertotalistic_CAs-resolution7.pdf")